# Generative Models: Mathematical Foundations

> *"A generative model is a compressed description of a universe — it must encode not just what exists, but the probability that each possibility exists, so that when you sample from it, you draw from the same distribution that produced reality."*

Interactive theory notebook covering VAEs, normalizing flows, GANs, diffusion models, flow matching, and evaluation metrics. Each section includes derivations, visualisations, and numerical verification.

In [ ]:
import numpy as np
import scipy.linalg as la
from scipy import stats
from scipy.special import logsumexp

try:
    import matplotlib.pyplot as plt
    import matplotlib
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 6]
    plt.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)
print('Setup complete.')


---

## 1. Taxonomy of Generative Models

Before diving into specific models, we visualise the key tradeoffs: sample quality, likelihood tractability, and training stability.

In [ ]:
# === 1.4 Taxonomy Visualisation ===

models = ['Autoregressive', 'Normalizing\nFlow', 'VAE', 'GAN', 'Diffusion', 'Flow\nMatching']
likelihood  = [5, 5, 3, 0, 3, 2]   # 0=none, 5=exact
quality     = [4, 3, 2, 4.5, 4.5, 4.5]
stability   = [5, 5, 5, 2, 5, 5]
speed       = [1, 4, 5, 5, 2, 3]   # inference speed

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = np.arange(len(models))
    width = 0.2
    colors = ['#440154', '#31688e', '#35b779', '#fde725']

    ax = axes[0]
    ax.bar(x - 1.5*width, likelihood, width, label='Likelihood', color=colors[0])
    ax.bar(x - 0.5*width, quality, width, label='Sample Quality', color=colors[1])
    ax.bar(x + 0.5*width, stability, width, label='Training Stability', color=colors[2])
    ax.bar(x + 1.5*width, speed, width, label='Inference Speed', color=colors[3])
    ax.set_xticks(x); ax.set_xticklabels(models, fontsize=10)
    ax.set_ylim(0, 6); ax.set_ylabel('Score (0-5)')
    ax.set_title('Generative Model Tradeoffs')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.scatter(quality, likelihood, s=200, c=range(len(models)), cmap='viridis', zorder=3)
    for i, m in enumerate(models):
        ax.annotate(m, (quality[i]+0.05, likelihood[i]+0.05), fontsize=9)
    ax.set_xlabel('Sample Quality'); ax.set_ylabel('Likelihood Tractability')
    ax.set_title('Quality vs Likelihood Tradeoff')
    ax.set_xlim(1, 6); ax.set_ylim(-0.5, 6)

    plt.tight_layout(); plt.show()
    print('Figure: Generative model comparison across 4 dimensions')
else:
    print('Model comparison: AR(likelihood=5,quality=4), Flow(5,3), VAE(3,2), GAN(0,4.5), Diffusion(3,4.5)')


---

## 2. Maximum Likelihood and KL Minimisation

MLE is equivalent to minimising $D_{\mathrm{KL}}(p_{\text{data}} \| p_\theta)$.
We verify this numerically and visualise the forward/reverse KL difference.

In [ ]:
# === 2.1 MLE = KL Minimisation ===

# Data: mixture of two Gaussians
np.random.seed(42)
n = 5000
k = np.random.binomial(1, 0.4, n)
x_data = np.where(k, np.random.normal(-2, 0.8, n), np.random.normal(2, 1.0, n))

# Fit a single Gaussian by MLE
mu_mle = x_data.mean()
sigma_mle = x_data.std()
nll_mle = -stats.norm.logpdf(x_data, mu_mle, sigma_mle).mean()

# Compute KL divergence at the MLE solution
x_grid = np.linspace(-6, 6, 1000)
p_data_hist, bins = np.histogram(x_data, bins=100, density=True)
bin_centers = (bins[:-1] + bins[1:]) / 2
p_model = stats.norm.pdf(bin_centers, mu_mle, sigma_mle)

print(f'MLE fit: mu={mu_mle:.4f}, sigma={sigma_mle:.4f}')
print(f'Mean NLL (MLE): {nll_mle:.4f}')
print(f'Lower bound (entropy): {stats.entropy(p_data_hist + 1e-10) / np.log(2):.4f} bits')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(x_data, bins=60, density=True, alpha=0.5, color='steelblue', label='Data (bimodal)')
    ax.plot(x_grid, stats.norm.pdf(x_grid, mu_mle, sigma_mle),
            'r-', lw=2, label=f'MLE Gaussian (mu={mu_mle:.2f})')
    ax.set_title('MLE fits the best single Gaussian (minimising forward KL)')
    ax.set_xlabel('x'); ax.set_ylabel('Density'); ax.legend()
    plt.tight_layout(); plt.show()
print('MLE covers both modes but is diffuse — forward KL must cover all mass')


In [ ]:
# === 2.1 Forward KL (mode-covering) vs Reverse KL (mode-seeking) ===

np.random.seed(42)
x_grid = np.linspace(-6, 6, 1000)

# True bimodal p_data
def p_data_pdf(x):
    return 0.4 * stats.norm.pdf(x, -2, 0.8) + 0.6 * stats.norm.pdf(x, 2, 1.0)

# Forward KL min: fit Gaussian to p_data by MLE
mu_fwd, sigma_fwd = -0.4, 2.0  # approximate MLE

# Reverse KL min: Gaussian covering just one mode
mu_rev1, sigma_rev1 = -2.0, 0.8  # mode 1
mu_rev2, sigma_rev2 =  2.0, 1.0  # mode 2

p_true = p_data_pdf(x_grid)
p_fwd  = stats.norm.pdf(x_grid, mu_fwd, sigma_fwd)
p_rev1 = stats.norm.pdf(x_grid, mu_rev1, sigma_rev1)

def kl_discrete(p, q, dx=x_grid[1]-x_grid[0]):
    mask = (p > 1e-10)
    return np.sum(p[mask] * np.log(p[mask] / (q[mask] + 1e-20))) * dx

print(f'KL(data||fwd Gaussian):  {kl_discrete(p_true, p_fwd):.4f}  (mode-covering, higher)')
print(f'KL(mode1||data):         {kl_discrete(p_rev1, p_true):.4f}  (mode-seeking, lower)')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (mu, sig, title, color) in zip(axes, [
        (mu_fwd, sigma_fwd, 'Forward KL min: mode-covering', '#31688e'),
        (mu_rev1, sigma_rev1, 'Reverse KL min: mode-seeking', '#d62728')]):
        ax.plot(x_grid, p_true, 'k-', lw=2, label='p_data')
        ax.plot(x_grid, stats.norm.pdf(x_grid, mu, sig), '-', lw=2, color=color, label='model')
        ax.fill_between(x_grid, stats.norm.pdf(x_grid, mu, sig), alpha=0.3, color=color)
        ax.set_title(title); ax.legend(); ax.set_xlabel('x'); ax.set_ylabel('Density')
    plt.tight_layout(); plt.show()
print('Conclusion: VAEs/diffusion ~ forward KL (cover all modes). GANs ~ reverse KL (sharp, mode-seeking).')


---

## 3. Variational Autoencoders

We derive the ELBO numerically, implement the reparameterisation trick, and verify the closed-form KL for Gaussian encoders.

In [ ]:
# === 3.1 ELBO Derivation: Numerical Verification ===

np.random.seed(42)

# VAE with 2D latent, 1D data
# True p(x|z) = N(z, 0.1^2), prior p(z) = N(0,1)
# Encoder q(z|x) = N(x, 0.5^2) -- approximate posterior

x_obs = 1.5  # observed data point
mu_enc, sig_enc = x_obs * 0.9, 0.5  # encoder output
sig_dec = 0.1  # decoder noise

# Monte Carlo ELBO estimate
n_samples = 50000
eps = np.random.randn(n_samples)
z_samples = mu_enc + sig_enc * eps  # reparameterisation

# Reconstruction term E_q[log p(x|z)]
log_p_x_given_z = stats.norm.logpdf(x_obs, z_samples, sig_dec)
recon = log_p_x_given_z.mean()

# KL term (closed form)
kl_cf = 0.5 * (mu_enc**2 + sig_enc**2 - np.log(sig_enc**2) - 1)

# KL term (Monte Carlo)
log_q = stats.norm.logpdf(z_samples, mu_enc, sig_enc)
log_p = stats.norm.logpdf(z_samples, 0, 1)
kl_mc = (log_q - log_p).mean()

elbo = recon - kl_cf

print(f'Encoder: q(z|x) = N({mu_enc:.2f}, {sig_enc:.2f}^2)')
print(f'Reconstruction E_q[log p(x|z)]: {recon:.4f}')
print(f'KL (closed form):               {kl_cf:.4f}')
print(f'KL (Monte Carlo):               {kl_mc:.4f}')
print(f'ELBO:                           {elbo:.4f}')
ok = abs(kl_cf - kl_mc) < 0.01
print(f"{'PASS' if ok else 'FAIL'} — closed-form KL matches Monte Carlo")


In [ ]:
# === 3.2 Reparameterisation vs Score Function Gradient Variance ===

np.random.seed(42)

mu = 1.0
log_sigma = 0.0  # sigma = 1
sigma = np.exp(log_sigma)

def f(z): return z**2  # simple function to differentiate

n_trials = 10000

# Reparameterisation gradient: d/d_mu E[f(mu + sigma*eps)]
# = E[f'(mu + sigma*eps)] = E[2z]
eps = np.random.randn(n_trials)
z = mu + sigma * eps
reparam_grads = 2 * z  # df/dz = 2z, dz/dmu = 1

# Score function gradient: E[f(z) * d/d_mu log q(z)]
# = E[z^2 * (z - mu) / sigma^2]
score_grads = f(z) * (z - mu) / sigma**2

print('Gradient of E_q[z^2] w.r.t. mu (true = 2*mu = 2.0):')
print(f'  Reparameterisation: mean={reparam_grads.mean():.4f}, std={reparam_grads.std():.4f}')
print(f'  Score function:     mean={score_grads.mean():.4f}, std={score_grads.std():.4f}')
print(f'  Variance ratio (score/reparam): {score_grads.var()/reparam_grads.var():.1f}x')
ok_reparam = abs(reparam_grads.mean() - 2.0) < 0.05
print(f"{'PASS' if ok_reparam else 'FAIL'} — reparameterisation gradient is accurate")


In [ ]:
# === 3.3 KL Geometry: Closed-Form Gaussian KL ===

np.random.seed(42)

def kl_gaussian_standard(mu, log_var):
    """KL(N(mu, diag(exp(log_var))) || N(0, I))"""
    return 0.5 * np.sum(mu**2 + np.exp(log_var) - log_var - 1)

# Visualise KL surface as function of mu and sigma
mu_vals = np.linspace(-3, 3, 50)
sig_vals = np.linspace(0.1, 3, 50)
MU, SIG = np.meshgrid(mu_vals, sig_vals)
KL = 0.5 * (MU**2 + SIG**2 - 2*np.log(SIG) - 1)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 6))
    cs = ax.contourf(MU, SIG, KL, levels=20, cmap='viridis')
    ax.contour(MU, SIG, KL, levels=[0.1, 0.5, 1, 2, 5], colors='white', alpha=0.5)
    plt.colorbar(cs, ax=ax, label='KL divergence')
    ax.plot(0, 1, 'r*', markersize=15, label='KL=0 (mu=0, sigma=1)')
    ax.set_xlabel('mu'); ax.set_ylabel('sigma')
    ax.set_title('KL(N(mu,sigma^2) || N(0,1)): minimum at (0,1)')
    ax.legend(); plt.tight_layout(); plt.show()

# Verify KL = 0 at prior
kl_at_prior = kl_gaussian_standard(np.array([0.0]), np.array([0.0]))
print(f'KL at prior (mu=0, log_var=0): {kl_at_prior:.8f}')
print(f"{'PASS' if abs(kl_at_prior) < 1e-10 else 'FAIL'} — KL = 0 at prior")

# KL for mu=2, sigma=0.5
kl_ex = kl_gaussian_standard(np.array([2.0]), np.array([2*np.log(0.5)]))
print(f'KL(N(2, 0.25) || N(0,1)) = {kl_ex:.4f}')


---

## 4. Normalizing Flows

We implement a RealNVP coupling layer, verify its Jacobian, and compute exact likelihoods.

In [ ]:
# === 4.3 RealNVP Coupling Layer: Jacobian and Invertibility ===

np.random.seed(42)
d = 4  # latent dimension

# Simple linear scale/shift networks
Ws = np.random.randn(d//2, d//2) * 0.2
Wt = np.random.randn(d//2, d//2) * 0.2

def scale_net(z1): return z1 @ Ws  # s(z_1:d/2)
def shift_net(z1): return z1 @ Wt  # t(z_1:d/2)

def coupling_forward(z):
    """Affine coupling: y_a = z_a, y_b = z_b * exp(s(z_a)) + t(z_a)"""
    z1, z2 = z[:, :d//2], z[:, d//2:]
    s = scale_net(z1); t = shift_net(z1)
    y2 = z2 * np.exp(s) + t
    return np.concatenate([z1, y2], axis=1)

def coupling_inverse(y):
    """Inverse: z_a = y_a, z_b = (y_b - t(y_a)) * exp(-s(y_a))"""
    y1, y2 = y[:, :d//2], y[:, d//2:]
    s = scale_net(y1); t = shift_net(y1)
    z2 = (y2 - t) * np.exp(-s)
    return np.concatenate([y1, z2], axis=1)

def log_det_jacobian(z):
    """Log |det J| = sum of scale outputs"""
    z1 = z[:, :d//2]
    return scale_net(z1).sum(axis=1)

# Test
np.random.seed(0)
z_test = np.random.randn(5, d)
y_test = coupling_forward(z_test)
z_rec  = coupling_inverse(y_test)
cycle_err = np.abs(z_rec - z_test).max()

print(f'Cycle consistency error (max): {cycle_err:.2e}')
print(f"{'PASS' if cycle_err < 1e-12 else 'FAIL'} — forward/inverse are consistent")

# Verify log-det by numerical Jacobian
z0 = z_test[0:1]
J_numerical = np.zeros((d, d))
eps = 1e-5
for i in range(d):
    dz = np.zeros((1, d)); dz[0, i] = eps
    J_numerical[:, i] = (coupling_forward(z0 + dz) - coupling_forward(z0 - dz)).flatten() / (2*eps)

log_det_numerical = np.log(abs(np.linalg.det(J_numerical)))
log_det_analytic  = log_det_jacobian(z0)[0]
print(f'Log-det (numerical Jacobian): {log_det_numerical:.6f}')
print(f'Log-det (analytic sum of s):  {log_det_analytic:.6f}')
ok = abs(log_det_numerical - log_det_analytic) < 1e-4
print(f"{'PASS' if ok else 'FAIL'} — analytic log-det matches numerical")


In [ ]:
# === 4.1 Exact Density Evaluation with Flows ===

np.random.seed(42)

def log_normal(x):
    """Standard normal log-density"""
    d = x.shape[1]
    return -0.5 * (x**2).sum(1) - 0.5 * d * np.log(2*np.pi)

def flow_log_prob(x):
    """log p_X(x) = log p_Z(f^{-1}(x)) + log|det J_{f^{-1}}|"""
    z = coupling_inverse(x)
    # Note: log|det J_{f^{-1}}| = -log|det J_f| evaluated at z
    return log_normal(z) - log_det_jacobian(z)

# Sample from flow: z ~ N(0,I), x = f(z)
n_samp = 1000
z_samp = np.random.randn(n_samp, d)
x_samp = coupling_forward(z_samp)

# Evaluate density at samples
log_px = flow_log_prob(x_samp)
print(f'Mean log p_X(x): {log_px.mean():.4f}')
print(f'Std  log p_X(x): {log_px.std():.4f}')
print(f'All finite: {np.isfinite(log_px).all()}')
print(f"PASS" if np.isfinite(log_px).all() else "FAIL", "— log probabilities are finite")

# Verify: log-prob under flow should be close to log-prob under base
# (since our coupling layers are close to identity for small weights)
log_pz_at_z = log_normal(z_samp)
print(f'Mean log p_Z(z): {log_pz_at_z.mean():.4f}')
print(f'Difference: {abs(log_px.mean() - log_pz_at_z.mean()):.4f}')


---

## 5. Generative Adversarial Networks

We derive the optimal GAN discriminator and show the game objective equals JSD.

In [ ]:
# === 5.2 Optimal Discriminator and JSD ===

np.random.seed(42)

# p_data = N(0, 1), p_g = N(2, 1)
mu_r, sigma_r = 0.0, 1.0
mu_g, sigma_g = 2.0, 1.0

x_grid = np.linspace(-5, 7, 1000)
p_r = stats.norm.pdf(x_grid, mu_r, sigma_r)
p_g = stats.norm.pdf(x_grid, mu_g, sigma_g)

# Optimal discriminator D*(x) = p_r / (p_r + p_g)
D_star = p_r / (p_r + p_g + 1e-30)

# JSD via optimal discriminator objective
dx = x_grid[1] - x_grid[0]
V_Dstar = (p_r * np.log(D_star + 1e-30) + p_g * np.log(1 - D_star + 1e-30)).sum() * dx

# JSD directly
m = (p_r + p_g) / 2
kl_p_m = (p_r * np.log(p_r / (m + 1e-30) + 1e-30)).sum() * dx
kl_q_m = (p_g * np.log(p_g / (m + 1e-30) + 1e-30)).sum() * dx
jsd = 0.5 * (kl_p_m + kl_q_m)

print(f'V(D*, G) = {V_Dstar:.4f}')
print(f'-log(4) + 2*JSD = {-np.log(4) + 2*jsd:.4f}')
ok = abs(V_Dstar - (-np.log(4) + 2*jsd)) < 0.01
print(f"{'PASS' if ok else 'FAIL'} — V(D*,G) = -log4 + 2*JSD")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(x_grid, p_r, 'b-', label='p_data ~ N(0,1)')
    axes[0].plot(x_grid, p_g, 'r-', label='p_g ~ N(2,1)')
    axes[0].fill_between(x_grid, p_r, alpha=0.2, color='blue')
    axes[0].fill_between(x_grid, p_g, alpha=0.2, color='red')
    axes[0].set_title('Real vs Generated Distribution')
    axes[0].set_xlabel('x'); axes[0].set_ylabel('Density'); axes[0].legend()

    axes[1].plot(x_grid, D_star, 'g-', lw=2, label='D*(x)')
    axes[1].axhline(0.5, color='gray', ls='--', label='D=0.5 (equilibrium)')
    axes[1].set_title('Optimal Discriminator D*(x) = p_r/(p_r+p_g)')
    axes[1].set_xlabel('x'); axes[1].set_ylabel('D*(x)'); axes[1].legend()
    plt.tight_layout(); plt.show()


---

## 6. Diffusion Models

Diffusion models define a **forward noising process** that gradually destroys data,
then learn a **reverse denoising process** to reconstruct it.

**Forward marginal** (DDPM, Ho et al. 2020):
$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

where $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ and $\alpha_s = 1-\beta_s$.

**Simplified training objective** (noise prediction):
$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t,\mathbf{x}_0,\boldsymbol{\epsilon}}\left[\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\sqrt{\bar{\alpha}_t}\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\boldsymbol{\epsilon},\, t)\|^2\right]$$

**Score connection**: $\boldsymbol{\epsilon}_\theta \approx -\sqrt{1-\bar{\alpha}_t}\,\nabla_{\mathbf{x}_t}\log p(\mathbf{x}_t)$

In [ ]:
# === 6.1 Forward Diffusion Process ===
import numpy as np
import json

try:
    import matplotlib.pyplot as plt
    import matplotlib
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.random.seed(42)

# Linear beta schedule
T = 1000
beta_start, beta_end = 1e-4, 0.02
betas = np.linspace(beta_start, beta_end, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)  # \bar{alpha}_t

# Forward marginal: q(x_t | x_0) = N(sqrt(abar)*x0, (1-abar)*I)
def q_sample(x0, t):
    """Sample x_t given x_0 via reparameterisation."""
    abar = alpha_bar[t]
    eps = np.random.randn(*x0.shape)
    return np.sqrt(abar) * x0 + np.sqrt(1 - abar) * eps, eps

# Visualize signal degradation
x0 = np.sin(np.linspace(0, 2*np.pi, 64))  # toy 1-D signal
timesteps_show = [0, 100, 250, 500, 750, 999]

print('Forward diffusion: signal-to-noise ratio at key timesteps')
print(f'{'t':>6}  {'sqrt(abar)':>12}  {'sqrt(1-abar)':>14}  {'SNR':>10}')
for t in timesteps_show:
    ab = alpha_bar[t]
    snr = ab / (1 - ab)
    print(f'{t:>6}  {np.sqrt(ab):>12.6f}  {np.sqrt(1-ab):>14.6f}  {snr:>10.4f}')

if HAS_MPL:
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    for ax, t in zip(axes.flat, timesteps_show):
        xt, _ = q_sample(x0, t)
        ax.plot(xt, color='steelblue', linewidth=1.2)
        ax.set_title(f't={t},  SNR={alpha_bar[t]/(1-alpha_bar[t]):.3f}')
        ax.set_ylim(-3.5, 3.5)
        ax.axhline(0, color='gray', linewidth=0.5)
    plt.suptitle('Forward Diffusion: Signal Degradation Over Time', fontsize=14)
    plt.tight_layout()
    plt.savefig('/tmp/diffusion_forward.png', dpi=100)
    plt.show()
    print('Figure saved.')

# Verify: at t=999, x_t should be near pure noise
xT, _ = q_sample(x0, 999)
print(f'\nAt T=999: mean(|x_T - 0|) = {np.mean(np.abs(xT)):.4f}')
print(f'sqrt(alpha_bar[999]) = {np.sqrt(alpha_bar[999]):.6f}  (signal coeff ~ 0)')
ok = np.sqrt(alpha_bar[999]) < 0.05
print(f"{'PASS' if ok else 'FAIL'} — signal is nearly gone at T")

In [ ]:
# === 6.2 ELBO Decomposition and Simplified Loss ===

# DDPM ELBO (full form):
#   L = L_T  +  sum_{t=2}^{T} L_{t-1}  +  L_0
# L_{t-1} = KL[ q(x_{t-1}|x_t,x0) || p_theta(x_{t-1}|x_t) ]
#
# Optimal reverse mean (posterior q):
#   mu_tilde = (sqrt(abar_{t-1})*beta_t / (1-abar_t)) * x0
#            + (sqrt(alpha_t)*(1-abar_{t-1}) / (1-abar_t)) * x_t

def posterior_mean(x0, xt, t):
    """True posterior mean tilde_mu(x_t, x_0)."""
    ab   = alpha_bar[t]
    ab1  = alpha_bar[t-1] if t > 0 else 1.0
    beta = betas[t]
    a    = alphas[t]
    coef_x0 = np.sqrt(ab1) * beta / (1.0 - ab)
    coef_xt = np.sqrt(a) * (1.0 - ab1) / (1.0 - ab)
    return coef_x0 * x0 + coef_xt * xt

# Monte-Carlo verification: posterior variance
#  sigma_t^2 = (1 - abar_{t-1}) / (1 - abar_t) * beta_t
def posterior_var(t):
    ab1 = alpha_bar[t-1] if t > 0 else 1.0
    return (1.0 - ab1) / (1.0 - alpha_bar[t]) * betas[t]

# Verify at t=500
t0 = 500
x0_test = np.array([1.0])
N = 50000
samples_t = np.array([q_sample(x0_test, t0)[0] for _ in range(N)]).squeeze()
samples_tm1 = np.array([q_sample(x0_test, t0-1)[0] for _ in range(N)]).squeeze()

# Empirical posterior: q(x_{t-1}|x_t=fixed, x0)
# Use a fixed x_t value near the mean
xt_fixed = np.sqrt(alpha_bar[t0]) * x0_test[0]
mu_tilde = posterior_mean(x0_test[0], xt_fixed, t0)
sig2_tilde = posterior_var(t0)

print(f'Posterior at t={t0}:')
print(f'  tilde_mu   = {mu_tilde:.6f}')
print(f'  tilde_sig2 = {sig2_tilde:.8f}')
print(f'  tilde_sig  = {np.sqrt(sig2_tilde):.8f}')

# Simplified loss: predict epsilon
# If eps_theta = eps_true, L_simple = 0
def simplified_loss_sample(x0, t):
    xt, eps_true = q_sample(x0, t)
    # Mock perfect predictor
    eps_pred = eps_true
    return np.mean((eps_pred - eps_true)**2)

losses = [simplified_loss_sample(np.random.randn(64), np.random.randint(1, T)) for _ in range(100)]
print(f'\nSimplified loss (perfect predictor, 100 samples): {np.mean(losses):.6f}')
ok = np.mean(losses) < 1e-20
print(f"{'PASS' if ok else 'FAIL'} — perfect predictor achieves zero simplified loss")

In [ ]:
# === 6.3 DDIM Deterministic Sampling ===

# DDIM (Song et al. 2020): non-Markovian reverse process
# x_{t-1} = sqrt(abar_{t-1}) * x0_pred
#          + sqrt(1-abar_{t-1} - eta^2*sigma^2) * eps_pred
#          + eta * sigma * z
# where eta=0 gives deterministic, eta=1 recovers DDPM

def x0_from_xt_eps(xt, eps_pred, t):
    """Predict x0 from x_t and eps_pred."""
    ab = alpha_bar[t]
    return (xt - np.sqrt(1 - ab) * eps_pred) / np.sqrt(ab)

def ddim_step(xt, eps_pred, t, t_prev, eta=0.0):
    """One DDIM reverse step."""
    ab  = alpha_bar[t]
    ab1 = alpha_bar[t_prev] if t_prev >= 0 else 1.0
    x0_pred = x0_from_xt_eps(xt, eps_pred, t)
    sigma = eta * np.sqrt((1 - ab1) / (1 - ab)) * np.sqrt(1 - ab / ab1)
    dir_xt = np.sqrt(1 - ab1 - sigma**2) * eps_pred
    z = np.random.randn(*xt.shape) if eta > 0 else 0.0
    return np.sqrt(ab1) * x0_pred + dir_xt + sigma * z

# Simulate DDIM with 50 steps (stride 20) from noise
np.random.seed(0)
x_true = np.array([1.5, -0.5, 0.8])  # mock x0
xT = np.random.randn(3)

# Perfect eps predictor (oracle)
def oracle_eps(xt, t):
    ab = alpha_bar[t]
    return (xt - np.sqrt(ab) * x_true) / np.sqrt(1 - ab)

# DDIM 50-step sampling
stride = T // 50
timesteps = list(range(T-1, -1, -stride))
x = xT.copy()
for i, t in enumerate(timesteps[:-1]):
    t_prev = timesteps[i+1]
    eps = oracle_eps(x, t)
    x = ddim_step(x, eps, t, t_prev, eta=0.0)

print('DDIM 50-step reconstruction (eta=0, deterministic):')
print(f'  x_true      = {x_true}')
print(f'  x_recovered = {x}')
print(f'  max error   = {np.max(np.abs(x - x_true)):.6f}')
ok = np.allclose(x, x_true, atol=0.01)
print(f"{'PASS' if ok else 'FAIL'} — DDIM oracle recovery within 1%")

---

## 7. Score Matching and Energy-Based Models

**Score function**: $s(\mathbf{x}) = \nabla_{\mathbf{x}} \log p(\mathbf{x})$

**Implicit score matching** (Hyvärinen 2005) avoids the intractable partition function:
$$J_{\text{ISM}}(\theta) = \mathbb{E}_{p_\text{data}}\left[
\text{tr}(\nabla_{\mathbf{x}} s_\theta) + \tfrac{1}{2}\|s_\theta\|^2\right]$$

**Denoising Score Matching** (Vincent 2011):
$$J_{\text{DSM}}(\theta) = \mathbb{E}_{q_\sigma(\tilde{\mathbf{x}}|\mathbf{x})}\left[
\|s_\theta(\tilde{\mathbf{x}}, \sigma) - \nabla_{\tilde{\mathbf{x}}}\log q_\sigma(\tilde{\mathbf{x}}|\mathbf{x})\|^2\right]$$

**SDE Framework** (Song et al. 2021): unified view via forward SDE
$$d\mathbf{x} = f(\mathbf{x},t)\,dt + g(t)\,d\mathbf{w}$$
Reverse SDE: $d\mathbf{x} = [f - g^2 \nabla_{\mathbf{x}}\log p_t(\mathbf{x})]\,dt + g\,d\bar{\mathbf{w}}$

In [ ]:
# === 7.1 Denoising Score Matching ===

# DSM target: score of q_sigma(x_tilde | x) = N(x, sigma^2 I)
# grad_{x_tilde} log q = (x - x_tilde) / sigma^2

def dsm_target(x, x_tilde, sigma):
    """True DSM score target."""
    return (x - x_tilde) / (sigma**2)

# Verify: DSM == ISM (they are equal for Gaussian noise)
# At optimum, s_theta(x_tilde) = E[dsm_target | x_tilde]
# For Gaussian mixture data:
def gaussian_score(x, mu, sig):
    """Score of N(mu, sig^2)."""
    return -(x - mu) / sig**2

# Toy 1-D bimodal: p(x) = 0.5 N(-2,1) + 0.5 N(2,1)
# Analytic score: p'(x)/p(x)
def true_score(x):
    p1 = 0.5 * np.exp(-0.5*(x+2)**2) / np.sqrt(2*np.pi)
    p2 = 0.5 * np.exp(-0.5*(x-2)**2) / np.sqrt(2*np.pi)
    dp1 = p1 * (-(x+2))
    dp2 = p2 * (-(x-2))
    return (dp1 + dp2) / (p1 + p2)

xs = np.linspace(-6, 6, 300)
scores = true_score(xs)

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    p1 = 0.5*np.exp(-0.5*(xs+2)**2)/np.sqrt(2*np.pi)
    p2 = 0.5*np.exp(-0.5*(xs-2)**2)/np.sqrt(2*np.pi)
    ax1.plot(xs, p1+p2, 'steelblue', linewidth=2, label='p(x)')
    ax1.set_title('Bimodal Data Distribution')
    ax1.set_xlabel('x'); ax1.legend()
    ax2.plot(xs, scores, 'coral', linewidth=2, label='score s(x)')
    ax2.axhline(0, color='gray', linewidth=0.5)
    ax2.set_title('Score Function s(x) = ∇ log p(x)')
    ax2.set_xlabel('x'); ax2.legend()
    plt.tight_layout()
    plt.savefig('/tmp/score_function.png', dpi=100)
    plt.show()

# Verify score at modes: score should be zero at x = +/-2
print('Score at bimodal peaks:')
print(f'  score(-2) = {true_score(-2.0):.6f}  (expect ~0)')
print(f'  score(+2) = {true_score(+2.0):.6f}  (expect ~0)')
ok = abs(true_score(-2.0)) < 0.01 and abs(true_score(2.0)) < 0.01
print(f"{'PASS' if ok else 'FAIL'} — score is zero at distribution modes")

In [ ]:
# === 7.2 Langevin Dynamics Sampling ===

# Langevin MCMC:
#   x_{k+1} = x_k + (epsilon/2) * score(x_k) + sqrt(epsilon) * z_k
# Converges to p(x) as epsilon->0, K->inf

def langevin_sample(score_fn, x_init, eps=0.01, K=2000):
    """Unadjusted Langevin algorithm."""
    x = x_init.copy()
    trajectory = [x.copy()]
    for _ in range(K):
        grad = score_fn(x)
        noise = np.random.randn(*x.shape)
        x = x + (eps/2) * grad + np.sqrt(eps) * noise
        trajectory.append(x.copy())
    return np.array(trajectory)

# Sample from bimodal distribution
np.random.seed(1)
x_init = np.array([0.0])  # start at mode barrier
traj = langevin_sample(lambda x: true_score(x[0]) * np.ones(1),
                        x_init, eps=0.05, K=3000)

samples = traj[1000:, 0]  # discard burn-in
print(f'Langevin samples: mean={np.mean(samples):.3f}, std={np.std(samples):.3f}')
print(f'Expected: mean~0 (symmetric bimodal), std~2.2')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(traj[:, 0], alpha=0.7, linewidth=0.5, color='steelblue')
    ax1.set_title('Langevin Trajectory'); ax1.set_xlabel('Step')
    ax2.hist(samples, bins=50, density=True, alpha=0.6, color='steelblue')
    xs2 = np.linspace(-6, 6, 300)
    ax2.plot(xs2, 0.5*np.exp(-0.5*(xs2+2)**2)/np.sqrt(2*np.pi)
                + 0.5*np.exp(-0.5*(xs2-2)**2)/np.sqrt(2*np.pi),
             'coral', linewidth=2, label='true p(x)')
    ax2.set_title('Langevin Samples vs True Distribution'); ax2.legend()
    plt.tight_layout(); plt.show()

ok = abs(np.mean(samples)) < 0.5  # roughly symmetric
print(f"{'PASS' if ok else 'FAIL'} — Langevin samples roughly symmetric around 0")

---

## 8. Flow Matching

**Conditional Flow Matching** (Lipman et al. 2022): train a vector field $v_\theta$ to transport
noise $\pi_0$ to data $\pi_1$ via an ODE $\dot{\mathbf{x}} = v_\theta(\mathbf{x},t)$.

**CFM objective**:
$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t,\mathbf{x}_0,\mathbf{x}_1}\left[
\|v_\theta(\mathbf{x}_t, t) - (\mathbf{x}_1 - \mathbf{x}_0)\|^2\right]$$

where $\mathbf{x}_t = (1-t)\mathbf{x}_0 + t\mathbf{x}_1$ (OT-CFM straight path).

**Rectified Flows** (Liu et al. 2022): same idea, iteratively reflow to straighten trajectories.

**Advantage over diffusion**: straight training paths → fewer NFE at inference.

In [ ]:
# === 8.1 Flow Matching: Straight Paths ===

# OT-CFM: interpolant x_t = (1-t)*x0 + t*x1
# Target velocity: u_t(x|x0,x1) = x1 - x0  (constant!)

def ot_interpolant(x0, x1, t):
    return (1-t)*x0 + t*x1

def ot_velocity(x0, x1):
    return x1 - x0  # constant velocity field

# Toy: source = N(0,I), target = N(3,0.5^2)
np.random.seed(42)
N = 2000
x0_samp = np.random.randn(N)
x1_samp = 3.0 + 0.5*np.random.randn(N)

# Simulate OT-FM flow with Euler integration
def euler_ode(x_init, velocity_fn, steps=50):
    x = x_init.copy()
    dt = 1.0 / steps
    for i in range(steps):
        t = i * dt
        v = velocity_fn(x, t)
        x = x + dt * v
    return x

# Oracle velocity field (knows x0 and x1)
# In practice, we condition on x0 and predict x1
# Here: v(x,t) = (x1 - x0) for each pair
x0_test = np.random.randn(500)
x1_test = 3.0 + 0.5*np.random.randn(500)

def oracle_velocity(x, t, x0=x0_test, x1=x1_test):
    return x1 - x0  # true conditional velocity

x_final = euler_ode(x0_test, oracle_velocity, steps=100)
print(f'OT-FM transport result:')
print(f'  Source:  mean={x0_test.mean():.3f}, std={x0_test.std():.3f}')
print(f'  Target:  mean={x1_test.mean():.3f}, std={x1_test.std():.3f}')
print(f'  Mapped:  mean={x_final.mean():.3f}, std={x_final.std():.3f}')

ok = abs(x_final.mean() - 3.0) < 0.1 and abs(x_final.std() - 0.5) < 0.1
print(f"{'PASS' if ok else 'FAIL'} — flow correctly transports N(0,1) to N(3,0.5)")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for t_show, ax in zip([0.0, 0.5, 1.0], axes):
        xt = ot_interpolant(x0_samp, x1_samp, t_show)
        ax.hist(xt, bins=50, density=True, alpha=0.7, color='steelblue')
        ax.set_title(f't = {t_show}'); ax.set_xlim(-4, 6)
    axes[0].set_xlabel('source N(0,1)')
    axes[2].set_xlabel('target N(3,0.5)')
    plt.suptitle('OT-FM: Straight-Path Interpolation', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 9. Evaluation Metrics

### Fréchet Inception Distance (FID)
$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{tr}\left(\Sigma_r + \Sigma_g
- 2(\Sigma_r\Sigma_g)^{1/2}\right)$$

Lower is better. Requires computing the matrix square root $(\Sigma_r \Sigma_g)^{1/2}$
via eigendecomposition.

### Inception Score (IS)
$$\text{IS} = \exp\left(\mathbb{E}_{\mathbf{x}\sim p_g}\left[
D_{\text{KL}}(p(y|\mathbf{x}) \| p(y))\right]\right)$$

High IS ↔ sharp (high $p(y|\mathbf{x})$) AND diverse (flat $p(y)$).

### Bits Per Dimension (BPD)
$$\text{BPD} = -\frac{\log_2 p_\theta(\mathbf{x})}{D}$$

where $D$ is the data dimensionality. Used for exact-likelihood models.

In [ ]:
# === 9.1 FID Computation ===

from scipy.linalg import sqrtm

def compute_fid(mu_r, sigma_r, mu_g, sigma_g):
    """
    FID = ||mu_r - mu_g||^2 + tr(Sigma_r + Sigma_g - 2*(Sigma_r @ Sigma_g)^{1/2})
    """
    diff = mu_r - mu_g
    # Matrix square root via scipy
    covmean = sqrtm(sigma_r @ sigma_g)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff @ diff + np.trace(sigma_r + sigma_g - 2*covmean)
    return fid

# Test 1: identical distributions -> FID = 0
d = 10
np.random.seed(42)
mu = np.random.randn(d)
A = np.random.randn(d, d)
sigma = A @ A.T / d + np.eye(d)*0.1
fid_same = compute_fid(mu, sigma, mu, sigma)
print(f'FID (identical distributions): {fid_same:.8f}  (expect ~0)')
print(f"{'PASS' if abs(fid_same) < 1e-6 else 'FAIL'} — FID=0 for identical distributions")

# Test 2: shifted mean -> FID = ||delta_mu||^2
delta = np.ones(d) * 0.5
fid_shifted = compute_fid(mu, sigma, mu + delta, sigma)
fid_expected = delta @ delta
print(f'\nFID (shifted mean by 0.5): {fid_shifted:.6f}')
print(f'Expected (||delta||^2):    {fid_expected:.6f}')
ok = abs(fid_shifted - fid_expected) < 1e-6
print(f"{'PASS' if ok else 'FAIL'} — FID correctly reflects mean shift")

# Test 3: FID scaling
fid_vals = []
noise_levels = np.linspace(0, 3, 20)
for noise in noise_levels:
    mu_g = mu + noise * np.random.randn(d)
    fid_vals.append(compute_fid(mu, sigma, mu_g, sigma))

if HAS_MPL:
    plt.figure(figsize=(8, 4))
    plt.plot(noise_levels, fid_vals, 'steelblue', linewidth=2)
    plt.xlabel('Mean shift magnitude'); plt.ylabel('FID')
    plt.title('FID vs Distribution Divergence')
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print('FID increases monotonically with distribution divergence.')

In [ ]:
# === 9.2 Inception Score and BPD ===

# IS = exp(E_x[KL(p(y|x) || p(y))])
# = exp(H(p(y)) - E_x[H(p(y|x))])

def compute_is(py_given_x):
    """
    py_given_x: (N, C) softmax probabilities
    Returns: IS score
    """
    py = py_given_x.mean(axis=0)  # marginal
    eps = 1e-10
    kl = py_given_x * (np.log(py_given_x + eps) - np.log(py + eps))
    kl_mean = kl.sum(axis=1).mean()
    return np.exp(kl_mean)

C = 10  # classes
N = 1000  # generated samples

# Scenario 1: perfect quality + diversity (IS = C)
# Each sample is confident about a different class
py_perf = np.zeros((N, C))
for i in range(N):
    py_perf[i, i % C] = 1.0
is_perf = compute_is(py_perf + 1e-10)
print(f'IS (perfect, uniform over classes): {is_perf:.4f}  (expect ~{C})')

# Scenario 2: mode collapse (all samples -> class 0)
py_mode = np.zeros((N, C))
py_mode[:, 0] = 1.0
is_mode = compute_is(py_mode + 1e-10)
print(f'IS (mode collapse, 1 class):        {is_mode:.4f}  (expect ~1)')

# Scenario 3: low quality (uniform per sample)
py_low = np.ones((N, C)) / C
is_low = compute_is(py_low + 1e-10)
print(f'IS (uniform conditional):           {is_low:.4f}  (expect ~1)')

ok = is_perf > is_mode and is_perf > is_low
print(f"{'PASS' if ok else 'FAIL'} — IS correctly ranks quality+diversity")

# BPD: for a d-dim Gaussian
def gaussian_bpd(x, mu, sigma_sq, d):
    log_p = -0.5*(((x - mu)**2).sum() / sigma_sq + d*np.log(2*np.pi*sigma_sq))
    return -log_p / (d * np.log(2))

d = 64
x_test = np.random.randn(d)
bpd = gaussian_bpd(x_test, np.zeros(d), 1.0, d)
# Standard Gaussian BPD ≈ log2(e)/2 * (1 + log(2*pi)) ≈ 2.047
print(f'\nBPD of N(0,I) sample under N(0,I): {bpd:.4f}  (expect ~2.05)')
print(f"{'PASS' if 1.5 < bpd < 2.8 else 'FAIL'} — BPD in expected range for Gaussian")

---

## 10. Latent Diffusion and Modern Architectures

**Latent Diffusion Models** (Rombach et al. 2022): run diffusion in a compressed latent space.

$$z = \mathcal{E}(x), \quad \hat{x} = \mathcal{D}(z)$$

Diffusion operates on $z$ (e.g., $64\times 64 \times 4$ for $512^2$ images), reducing
compute by $8\times$–$16\times$ vs pixel space.

**U-Net with cross-attention**: denoising backbone $\epsilon_\theta(z_t, t, \tau_\theta(y))$
where $\tau_\theta$ is a text encoder (CLIP, T5). Cross-attention injects text conditioning.

**Classifier-free guidance** (Ho & Salimans 2021):
$$\hat{\epsilon} = \epsilon_\theta(z_t, t, \emptyset) + w\cdot[\epsilon_\theta(z_t, t, c)
- \epsilon_\theta(z_t, t, \emptyset)]$$

Higher $w$ → sharper, more condition-aligned images (but less diversity).

**DiT** (Peebles & Xie 2022): replace U-Net with a Vision Transformer. **Flux** (Black Forest Labs):
MMDiT architecture with separate text and image streams.

In [ ]:
# === 10.1 Classifier-Free Guidance ===

# CFG trades diversity for fidelity:
# eps_hat = eps_uncond + w*(eps_cond - eps_uncond)

# Simulate: conditional prediction shifts toward target
np.random.seed(42)

def cfg_sample(eps_cond, eps_uncond, w):
    return eps_uncond + w * (eps_cond - eps_uncond)

d = 32
# Uncond prediction: noise-like (mean ~0)
eps_uncond = np.random.randn(d) * 0.8
# Cond prediction: structured (mean ~2, smaller variance)
target_signal = np.ones(d) * 2.0
eps_cond = target_signal + np.random.randn(d) * 0.3

guidance_scales = [0.0, 1.0, 3.0, 7.5, 15.0]
print(f'CFG: eps_uncond.mean={eps_uncond.mean():.3f}, eps_cond.mean={eps_cond.mean():.3f}')
print(f'{'w':>6}  {'mean(eps_hat)':>16}  {'std(eps_hat)':>14}  Note')
for w in guidance_scales:
    eps_hat = cfg_sample(eps_cond, eps_uncond, w)
    note = 'uncond' if w == 0 else 'cond' if w == 1 else 'amplified'
    print(f'{w:>6.1f}  {eps_hat.mean():>16.4f}  {eps_hat.std():>14.4f}  {note}')

# Verify: w=0 -> uncond, w=1 -> cond
ok1 = np.allclose(cfg_sample(eps_cond, eps_uncond, 0.0), eps_uncond)
ok2 = np.allclose(cfg_sample(eps_cond, eps_uncond, 1.0), eps_cond)
print(f"{'PASS' if ok1 else 'FAIL'} — CFG w=0 recovers unconditional prediction")
print(f"{'PASS' if ok2 else 'FAIL'} — CFG w=1 recovers conditional prediction")

if HAS_MPL:
    ws = np.linspace(0, 15, 100)
    means = [cfg_sample(eps_cond, eps_uncond, w).mean() for w in ws]
    plt.figure(figsize=(8, 4))
    plt.plot(ws, means, 'steelblue', linewidth=2)
    plt.axhline(eps_uncond.mean(), linestyle='--', color='gray', label='uncond mean')
    plt.axhline(eps_cond.mean(), linestyle='--', color='coral', label='cond mean')
    plt.xlabel('Guidance scale w'); plt.ylabel('eps_hat mean')
    plt.title('Classifier-Free Guidance Effect'); plt.legend()
    plt.tight_layout(); plt.show()

---

## Summary

This notebook has covered the core mathematics of generative models:

| Section | Key Result | Code Verified |
|---------|-----------|---------------|
| Taxonomy | 6 generative families categorized | bar chart + scatter |
| MLE/KL | MLE = forward KL minimization | bimodal fit |
| VAEs | ELBO, reparam trick, KL geometry | Monte Carlo vs analytic |
| Normalizing Flows | RealNVP coupling, log-det | Jacobian numerical verify |
| GANs | Optimal discriminator, JSD connection | Nash equilibrium verify |
| Diffusion | Forward marginal, DDIM sampling | Oracle recovery |
| Score Matching | DSM, Langevin dynamics | Mode zeros, sampling |
| Flow Matching | OT-CFM straight paths | N(0,1)→N(3,0.5) transport |
| Evaluation | FID, IS, BPD | All verified analytically |
| Latent Diffusion | CFG, guidance scale effect | w=0/1 verified |

### Key Equations

| Model | Core Objective |
|-------|---------------|
| VAE | $\mathcal{L}_{\text{ELBO}} = \mathbb{E}_q[\log p_{\theta}(\mathbf{x}|\mathbf{z})] - D_{\text{KL}}(q_\phi \| p)$ |
| Flow | $\log p_\theta(\mathbf{x}) = \log p_0(f^{-1}(\mathbf{x})) + \log|\det J_{f^{-1}}|$ |
| GAN | $\min_G\max_D \;\mathbb{E}[\log D] + \mathbb{E}[\log(1-D(G(z)))]$ |
| Diffusion | $\mathcal{L}_{\text{simple}} = \mathbb{E}[\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t)\|^2]$ |
| Flow Matching | $\mathcal{L}_{\text{CFM}} = \mathbb{E}[\|v_\theta(\mathbf{x}_t,t) - (\mathbf{x}_1-\mathbf{x}_0)\|^2]$ |

**Next**: [Exercises notebook](exercises.ipynb) — 10 graded problems on these topics.